# 02 — Wearable Time Series Analysis
Exploring 90-day wearable signals and detecting deterioration patterns.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("pcp-panel-intelligence"):
        os.system("git clone https://github.com/Hannah-Hiltz/pcp-panel-intelligence.git")
    os.chdir("pcp-panel-intelligence")
    sys.path.insert(0, "src")
    os.system("pip install -q xgboost shap scikit-learn joblib")
    DATA_DIR  = "data/raw";      PROC_DIR  = "data/processed"
    FIG_DIR   = "reports/figures"; MODEL_DIR = "models"
    WEAR_DIR  = "data/raw/wearables"; PANEL_DIR = "data/raw/panel"
else:
    sys.path.insert(0, "../src")
    DATA_DIR  = "../data/raw";   PROC_DIR  = "../data/processed"
    FIG_DIR   = "../reports/figures"; MODEL_DIR = "../models"
    WEAR_DIR  = "../data/raw/wearables"; PANEL_DIR = "../data/raw/panel"

for d in [FIG_DIR, PROC_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
print(f"Environment: {'Colab' if IN_COLAB else 'Jupyter'}")


In [ ]:
df    = pd.read_csv(f"{PANEL_DIR}/patient_panel_flat.csv")
wear  = pd.read_csv(f"{WEAR_DIR}/wearable_timeseries.csv")
wear["date"] = pd.to_datetime(wear["date"])
print(f"Patients with wearables: {wear['patient_id'].nunique():,}")
print(f"Total wearable rows:     {len(wear):,}")
wear.head()


## Step count distribution — adherent vs non-adherent

In [ ]:
wear_meta = wear.merge(df[["patient_id","adherent","outreach_priority"]], on="patient_id")
fig, axes = plt.subplots(1,2, figsize=(13,5))
for adherent, label, color in [(True,"Adherent","#378ADD"),(False,"Non-adherent","#E24B4A")]:
    subset = wear_meta[wear_meta["adherent"]==adherent]["steps"].dropna()
    axes[0].hist(subset, bins=60, alpha=0.6, label=label, color=color)
axes[0].set_title("Daily step count distribution", fontsize=13)
axes[0].set_xlabel("Steps"); axes[0].set_ylabel("Frequency"); axes[0].legend()

daily_avg = wear_meta.groupby(["day_index","adherent"])["steps"].mean().reset_index()
for adherent, label, color in [(True,"Adherent","#378ADD"),(False,"Non-adherent","#E24B4A")]:
    subset = daily_avg[daily_avg["adherent"]==adherent]
    axes[1].plot(subset["day_index"], subset["steps"], label=label, color=color, alpha=0.8)
axes[1].set_title("Mean daily steps over 90 days", fontsize=13)
axes[1].set_xlabel("Day"); axes[1].set_ylabel("Mean steps"); axes[1].legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/step_distribution.png", dpi=150, bbox_inches="tight"); plt.show()


## Deterioration event detection — case study

In [ ]:
high_risk = df[df["outreach_priority"].isin(["Critical","High"]) & (df["has_wearable"]==1)]
if len(high_risk) > 0:
    example_pid = high_risk.iloc[0]["patient_id"]
    ex = wear[wear["patient_id"]==example_pid].sort_values("day_index")
    fig, axes = plt.subplots(3,1, figsize=(13,10), sharex=True)
    for ax, col, label, color in [
        (axes[0],"steps",   "Daily steps","#378ADD"),
        (axes[1],"resting_hr","Resting HR","#E24B4A"),
        (axes[2],"hrv",     "HRV","#97C459"),
    ]:
        ax.plot(ex["day_index"], ex[col], color=color, alpha=0.7, linewidth=1.2)
        roll = ex[col].rolling(7, min_periods=3).mean()
        ax.plot(ex["day_index"], roll, color=color, linewidth=2, label="7-day avg")
        ax.set_ylabel(label); ax.legend(fontsize=9)
        if ex[col].notna().any():
            low = ex["day_index"][ex[col] == ex[col].min()].values
            if len(low): ax.axvline(low[0], color="grey", linestyle="--", alpha=0.5, label="Nadir")
    axes[2].set_xlabel("Day (0 = study start)")
    axes[0].set_title(f"Wearable signal — {example_pid} (High/Critical priority patient)", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/deterioration_case.png", dpi=150, bbox_inches="tight"); plt.show()
    print(f"Patient: {example_pid}")
    print(f"Conditions: {df[df['patient_id']==example_pid]['n_conditions'].values[0]}")
    print(f"Deterioration window visible around day 55-75 for non-adherent patients")


## HRV trends by priority tier

In [ ]:
wear_meta = wear.merge(df[["patient_id","outreach_priority"]], on="patient_id")
fig, ax = plt.subplots(figsize=(13,5))
order  = ["Critical","High","Moderate","Routine"]
colors = ["#E24B4A","#EF9F27","#97C459","#5DCAA5"]
for tier, color in zip(order, colors):
    sub = wear_meta[wear_meta["outreach_priority"]==tier]
    daily = sub.groupby("day_index")["hrv"].mean()
    ax.plot(daily.index, daily.values, label=tier, color=color, linewidth=1.8, alpha=0.85)
ax.set_title("Mean HRV over 90 days by priority tier", fontsize=13)
ax.set_xlabel("Day"); ax.set_ylabel("Mean HRV"); ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/hrv_by_tier.png", dpi=150, bbox_inches="tight"); plt.show()
print("Key finding: Critical/High patients show declining HRV trend vs improving for Routine.")


## Wearable anomaly flag analysis

In [ ]:
print("Anomaly flag rate by priority tier:")
print(df.groupby("outreach_priority")["wear_anomaly_flag"].mean().mul(100).round(1).to_string())
print("\nMean step baseline deviation by tier:")
print(df.groupby("outreach_priority")["wear_steps_baseline_dev"].mean().round(3).to_string())
print("\nEquity: anomaly flag rate by income quintile:")
print(df.groupby("zip_income_quintile")["wear_anomaly_flag"].mean().mul(100).round(1).to_string())
print("\n(Patients without devices have wear_anomaly_flag=1 by conservative-upward design)")
